# a) Diagnose HC

In [14]:
import random

def first_choice_hc_with_trace(landscape, start):
    current = start
    visited = set([current])

    while True:
        current_value = landscape[current]
        moved = False
        equal_exists = False

        if current > 0:
            if landscape[current - 1] > current_value:
                current -= 1
                moved = True
            elif landscape[current - 1] == current_value:
                equal_exists = True

        if not moved and current < len(landscape) - 1:
            if landscape[current + 1] > current_value:
                current += 1
                moved = True
            elif landscape[current + 1] == current_value:
                equal_exists = True

        if current in visited:
            return current, "ridge"

        visited.add(current)

        if not moved:
            if equal_exists:
                return current, "plateau"
            else:
                return current, "local maximum"


def diagnose_hc(landscape, start):
    terminal, mode = first_choice_hc_with_trace(landscape, start)
    print(f"Terminated at state {terminal+1} with f={landscape[terminal]}. Failure mode: {mode}")

In [15]:
def run_diagnosis_tests():

    landscape1 = [1, 3, 2, 4, 1]
    landscape2 = [1, 3, 3, 3, 1]
    landscape3 = [1, 4, 2, 4, 1]

    print("Local Maximum Test:")
    diagnose_hc(landscape1, 1)

    print("\nPlateau Test:")
    diagnose_hc(landscape2, 2)

    print("\nRidge Test:")
    diagnose_hc(landscape3, 2)

# b) N-Queens

In [16]:
def count_conflicts(board):
    conflicts = 0
    n = len(board)

    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j] or abs(board[i] - board[j]) == abs(i - j):
                conflicts += 1

    return conflicts


def stochastic_hc_nqueens(board):
    current = board[:]
    current_conflicts = count_conflicts(current)

    while True:
        improved = False

        for _ in range(200):
            i, j = random.sample(range(len(board)), 2)

            new_board = current[:]
            new_board[i], new_board[j] = new_board[j], new_board[i]

            new_conflicts = count_conflicts(new_board)

            if new_conflicts < current_conflicts:
                current = new_board
                current_conflicts = new_conflicts
                improved = True
                break

        if not improved:
            break

    return current, current_conflicts


def solve_nqueens_rrhc(num_restarts):
    for r in range(1, num_restarts + 1):

        board = list(range(8))
        random.shuffle(board)

        solution, conflicts = stochastic_hc_nqueens(board)

        if conflicts == 0:
            return r, solution

    return None, None


def print_board(board):
    for r in range(8):
        row = ""
        for c in range(8):
            if board[c] == r:
                row += "Q "
            else:
                row += ". "
        print(row)

In [17]:
def run_nqueens():
    restarts, solution = solve_nqueens_rrhc(100)

    print("\nN-Queens Result:")
    print("Restarts needed:", restarts)
    print("Solution board:", solution)

    if solution:
        print("\nBoard:")
        print_board(solution)

# c) Benchmark

In [18]:
def benchmark():
    ks = [5, 10, 25, 50, 100]

    print("\nBenchmark Results:\n")

    for k in ks:
        success = 0
        total_restarts = 0

        for _ in range(30):
            r, sol = solve_nqueens_rrhc(k)

            if sol is not None:
                success += 1
                total_restarts += r

        success_rate = success / 30
        avg_restarts = (total_restarts / success) if success > 0 else 0

        print(f"k={k}, Success Rate={success_rate:.2f}, Avg Restarts={avg_restarts:.2f}")

# MAIN

In [19]:
def main():
    print("===== PART (a) =====")
    run_diagnosis_tests()

    print("\n===== PART (b) =====")
    run_nqueens()

    print("\n===== PART (c) =====")
    benchmark()

main()

===== PART (a) =====
Local Maximum Test:
Terminated at state 2 with f=3. Failure mode: ridge

Plateau Test:
Terminated at state 3 with f=3. Failure mode: ridge

Ridge Test:
Terminated at state 2 with f=4. Failure mode: ridge

===== PART (b) =====

N-Queens Result:
Restarts needed: 1
Solution board: [5, 0, 4, 1, 7, 2, 6, 3]

Board:
. Q . . . . . . 
. . . Q . . . . 
. . . . . Q . . 
. . . . . . . Q 
. . Q . . . . . 
Q . . . . . . . 
. . . . . . Q . 
. . . . Q . . . 

===== PART (c) =====

Benchmark Results:

k=5, Success Rate=0.90, Avg Restarts=2.41
k=10, Success Rate=1.00, Avg Restarts=2.97
k=25, Success Rate=1.00, Avg Restarts=2.70
k=50, Success Rate=1.00, Avg Restarts=2.60
k=100, Success Rate=1.00, Avg Restarts=2.37
